# Stockfish 19 no Colab
Abra em ambiente **CPU** (GPU não é usada pelo Stockfish). Execute as duas células em ordem.
Copie URL e chave para o painel do script Roblox existente. Estilo, agressividade e tempo continuam no painel.

## Execução pelo celular
A célula de início continua ativa enquanto atende às consultas. Isso não impede o Colab de encerrar o runtime.
Para continuar oficialmente após desconectar o navegador, confira o recurso de execução em segundo plano da sua assinatura (Pro+ com saldo suficiente; condições em https://research.google.com/colaboratory/faq.html).
No gratuito, não há garantia de continuidade ao trocar de aplicativo. Não são usados cliques automáticos ou mecanismos para burlar limites.
Um túnel torna a API acessível; não prolonga a vida da máquina. Confira as regras de uso de serviços externos antes de hospedar.
Ao parar a célula, servidor e túnel são encerrados. Ao apagar o runtime, instalação e chave são perdidas; uma nova sessão gera nova URL.


In [ ]:
# Instalar (execute uma vez por sessão).
import os, sys, subprocess, pathlib, urllib.request, time
from pathlib import Path
BASE = pathlib.Path("/content/chess-colab")
BASE.mkdir(exist_ok=True)
subprocess.run([sys.executable, "-m", "pip", "-q", "install", "chess==1.11.2"], check=True)
SERVER = BASE / "server.py"
urllib.request.urlretrieve("https://raw.githubusercontent.com/erickgustavo2025/chess/main/termux/server.py?t=" + str(time.time_ns()), SERVER)
ENGINE = BASE / "stockfish-src" / "src" / "stockfish"
PGO_MARKER = BASE / "pgo-v1.done"
if not ENGINE.exists() or not PGO_MARKER.exists():
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "-qq", "git", "g++", "make"], check=True)
    source = BASE / "stockfish-src"
    if not source.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", "sf_19",
                        "https://github.com/official-stockfish/Stockfish.git", str(source)], check=True)
    flags = pathlib.Path("/proc/cpuinfo").read_text()
    arch = "x86-64-avx2" if "avx2" in flags else "x86-64"
    subprocess.run(["make", "-j2", "profile-build", "ARCH=" + arch, "COMP=gcc"],
                   cwd=source / "src", check=True)
    PGO_MARKER.write_text("Stockfish 19 PGO\n")
TUNNEL = BASE / "cloudflared"
if not TUNNEL.exists():
    urllib.request.urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        TUNNEL)
    TUNNEL.chmod(0o755)
print("Instalado. Execute a próxima célula para iniciar.")


In [ ]:
# Iniciar: esta célula permanece executando enquanto o servidor estiver ligado.
#@title Configuração e início
THREADS = min(16, os.cpu_count() or 1)
HASH_MB = 1024
import time, json, secrets, threading, queue, re, socket
with socket.socket() as sock:
    sock.bind(("127.0.0.1", 0))
    PORT = sock.getsockname()[1]
LOCAL_URL = f"http://127.0.0.1:{PORT}"
if not 1 <= THREADS <= min(16, os.cpu_count() or 1):
    raise ValueError("Threads devem estar entre 1 e o total disponível: " + str(os.cpu_count()))
if not 16 <= HASH_MB <= 65536:
    raise ValueError("Hash deve estar entre 16 e 65536 MB")
key_file = BASE / ".api-key"
if not key_file.exists():
    key_file.write_text(secrets.token_urlsafe(32))
key_file.chmod(0o600)
key = key_file.read_text().strip()
server = tunnel = None
output = queue.Queue()
def collect(process, name):
    for line in process.stdout:
        output.put((name, line.rstrip()))
try:
    server = subprocess.Popen(
        [sys.executable, "-u", str(SERVER), "--engine", str(ENGINE),
         "--port", str(PORT), "--force-precise", "--adaptive-time", "--threads", str(THREADS), "--hash", str(HASH_MB)],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    threading.Thread(target=collect, args=(server, "motor"), daemon=True).start()
    for attempt in range(60):
        if server.poll() is not None:
            raise RuntimeError("Servidor encerrou; confira as mensagens abaixo")
        try:
            req = urllib.request.Request(LOCAL_URL + "/health",
                                         headers={"Authorization": "Bearer " + key})
            with urllib.request.urlopen(req, timeout=2) as response:
                health = json.load(response)
            break
        except Exception:
            time.sleep(1)
    else:
        raise RuntimeError("Servidor não respondeu em 60 segundos")
    print("Motor:", health["engine"].get("name"))
    print("Threads:", THREADS, "| Hash:", HASH_MB, "MB")
    print("CHAVE PARA O PAINEL:", key)
    tunnel = subprocess.Popen(
        [str(TUNNEL), "tunnel", "--protocol", "http2", "--url", LOCAL_URL],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    threading.Thread(target=collect, args=(tunnel, "túnel"), daemon=True).start()
    print("Aguardando URL HTTPS...")
    while server.poll() is None and tunnel.poll() is None:
        try:
            name, line = output.get(timeout=1)
        except queue.Empty:
            continue
        if "Chave para colar" in line:
            continue
        match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
        if match:
            print("\nURL PARA O PAINEL:", match.group(0), "\n", flush=True)
        elif name == "motor" or "ERR" in line or "Registered tunnel" in line:
            print(line, flush=True)
    raise RuntimeError("Motor ou túnel encerrou. Execute esta célula novamente.")
except KeyboardInterrupt:
    print("Parando servidor e túnel.")
finally:
    while not output.empty():
        name, line = output.get_nowait()
        if "Chave para colar" not in line:
            print(line)
    for process in (tunnel, server):
        if process and process.poll() is None:
            process.terminate()
            try:
                process.wait(timeout=5)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
